# Demo: Ingestion de Documentos

Este notebook demuestra cómo procesar documentos y construir un grafo de conocimiento.

In [10]:
import sys
from dotenv import load_dotenv
sys.path.append('..')
load_dotenv("../env/.env")

from graphrag.graph.neo4j_manager import Neo4jManager
from graphrag.ingestion.text_processor import TextProcessor

## 1. Inicializar Neo4j

In [11]:
neo4j = Neo4jManager()
neo4j.create_constraints()
neo4j.create_vector_index()

In [15]:
animals = [
    "Tiger", "Lion", "Elephant", "Dolphin", "Giant Panda", "Horse", "Penguin", "Wolf", "Shark", "Rabbit",
    "Gorilla", "Giraffe", "Cheetah", "Polar Bear", "Hippopotamus", "Zebra", "Red Panda", "Kangaroo", "Koala", "Sloth",
    "Meerkat", "Rhinoceros", "Snow Leopard", "Orangutan", "Chimpanzee", "Platypus", "Lemur", "Capybara", "Sea Otter", "Arctic Fox",
    "Jaguar", "Black Panther", "Brown Bear", "Okapi", "Tasmanian Devil", "Wombat", "Quokka", "Hedgehog", "Fennec Fox", "Bald Eagle",
    "Peacock", "Flamingo", "Parrot", "Hummingbird", "Swan", "Toucan", "Owl", "Puffin", "Woodpecker", "Raven",
    "Falcon", "Ostrich", "Emu", "Albatross", "Robin", "Blue Jay", "Cardinal", "Canary", "Guinea Pig", "Hamster",
    "Cow", "Pig", "Sheep", "Goat", "Chicken", "Duck", "Donkey", "Llama", "Alpaca", "Ferret",
    "Chinchilla", "Great White Shark", "Blue Whale", "Killer Whale (Orca)", "Sea Turtle", "Octopus", "Seahorse", "Clownfish", "Axolotl", "Manatee",
    "Stingray", "Walrus", "Seal", "Narwhal", "Lobster", "Jellyfish", "Chameleon", "Komodo Dragon", "King Cobra", "Bearded Dragon",
    "Green Iguana", "Tree Frog", "Monarch Butterfly", "Honey Bee", "Praying Mantis", "Ladybug", "Raccoon", "Camel", "Squirrel", "Buffalo"
]

print(f"Total animals: {len(animals)}")
print(animals)

Total animals: 100
['Tiger', 'Lion', 'Elephant', 'Dolphin', 'Giant Panda', 'Horse', 'Penguin', 'Wolf', 'Shark', 'Rabbit', 'Gorilla', 'Giraffe', 'Cheetah', 'Polar Bear', 'Hippopotamus', 'Zebra', 'Red Panda', 'Kangaroo', 'Koala', 'Sloth', 'Meerkat', 'Rhinoceros', 'Snow Leopard', 'Orangutan', 'Chimpanzee', 'Platypus', 'Lemur', 'Capybara', 'Sea Otter', 'Arctic Fox', 'Jaguar', 'Black Panther', 'Brown Bear', 'Okapi', 'Tasmanian Devil', 'Wombat', 'Quokka', 'Hedgehog', 'Fennec Fox', 'Bald Eagle', 'Peacock', 'Flamingo', 'Parrot', 'Hummingbird', 'Swan', 'Toucan', 'Owl', 'Puffin', 'Woodpecker', 'Raven', 'Falcon', 'Ostrich', 'Emu', 'Albatross', 'Robin', 'Blue Jay', 'Cardinal', 'Canary', 'Guinea Pig', 'Hamster', 'Cow', 'Pig', 'Sheep', 'Goat', 'Chicken', 'Duck', 'Donkey', 'Llama', 'Alpaca', 'Ferret', 'Chinchilla', 'Great White Shark', 'Blue Whale', 'Killer Whale (Orca)', 'Sea Turtle', 'Octopus', 'Seahorse', 'Clownfish', 'Axolotl', 'Manatee', 'Stingray', 'Walrus', 'Seal', 'Narwhal', 'Lobster', 'Jelly

## 2. Cargar documento de ejemplo

In [12]:
FILE_PATH = "../data/lion_prueba.md"
with open(FILE_PATH, 'r', encoding='utf-8') as f:
    markdown_text = f.read()

print(markdown_text)

# Lion

## Lion Facts

The lion is Africa’s apex predator

The lion is one of the largest, strongest, and most powerful felines in the world, second only in size to the Siberian Tiger. They are the largest cats on the African continent.

While most big cats are solitary hunters, lions are incredibly sociable animals that live together in family groups called pride.

They are some of the world’s most popular animals:

## Scientific Name and Classification

The scientific name for lions is Panthera leo. The genus Panthera is of Greek origin and comprises big cat species such as tigers, lions, jaguars, and leopards that have the ability to roar. Leo is the Latin word for lion.

There are two types of lion subspecies. One is named Panthera leo melanochaita and lives across South and East Africa. The second lion subspecies has the scientific name Panther Leo and lives in West Africa, Central Africa, and Asia.

You may see references to African and Asiatic lions. Up until 2017, there were tw

## 3. Procesar documento

In [13]:
processor = TextProcessor(neo4j, chunk_size=800, chunk_overlap=80)

processor.process_document(
    text=markdown_text,
    document_id="lion",
    metadata={"title": "Lion", "source": "A-Z Animals"}
)

Documento dividido en 33 chunks


Procesando chunks:   9%|▉         | 3/33 [01:47<16:59, 34.00s/it]

Nueva especie añadida: Carnivore


Procesando chunks:  36%|███▋      | 12/33 [03:08<05:30, 15.74s/it]


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

## 4. Verificar el grafo

In [ ]:
# Contar nodos
stats = neo4j.execute_query("""
MATCH (n)
RETURN labels(n)[0] as label, count(*) as count
ORDER BY count DESC
""")

for stat in stats:
    print(f"{stat['label']}: {stat['count']}")

In [ ]:
# Ver algunas entidades
entities = neo4j.execute_query("""
MATCH (e:Entity)
RETURN e.name as name, e.type as type, e.summary as summary
LIMIT 10
""")

for entity in entities:
    print(f"\n{entity['name']} ({entity['type']})")
    print(f"  {entity['summary'][:100]}..." if entity['summary'] else "  No summary")

In [ ]:
# Ver algunas relaciones
relationships = neo4j.execute_query("""
MATCH (s:Entity)-[r:RELATIONSHIP]->(t:Entity)
RETURN s.name as source, r.type as type, t.name as target, r.summary as summary
LIMIT 10
""")

for rel in relationships:
    print(f"\n{rel['source']} -[{rel['type']}]-> {rel['target']}")
    print(f"  {rel['summary'][:100]}..." if rel['summary'] else "  No summary")

## 5. Cleanup (opcional)

In [ ]:
# Descomentar para limpiar la base de datos
# neo4j.execute_query("MATCH (n) DETACH DELETE n")
neo4j.close()